# Chapter 10 · Graph Neural Networks for Materials — Colab notebook

Back to the chapter: <https://dongzhaohe321418-lab.github.io/materials-simulation-handbook/ch10-gnn/>

We turn a crystal into a graph, wrap it in a PyTorch Geometric `Data` object, and run a single forward pass of a small CGCNN-style model. This is the core of the chapter's pipeline; the full study trains the same architecture on thousands of Materials Project structures.

This notebook is meant for **Google Colab** rather than the in-browser JupyterLite kernel, because it needs heavy packages (and, where noted, a GPU) that cannot run under Pyodide. Open it in Colab, and where the install cell mentions it, switch the runtime to a GPU via **Runtime -> Change runtime type -> GPU** before running the rest.


## Install

`torch-geometric` provides the graph data structures and message-passing primitives; `pymatgen` builds and analyses the crystal structures. `torch` comes pre-installed on Colab. A GPU is optional for a single forward pass but recommended once you start training.

In [ ]:
!pip install torch-geometric pymatgen


## Build a crystal and convert it to a graph

We take an SrTiO3 perovskite, find every neighbour within a cutoff radius with pymatgen, and read off the directed edge list and edge distances — the raw ingredients of a crystal graph.

In [ ]:
import numpy as np
import torch
from pymatgen.core import Structure, Lattice

lattice = Lattice.cubic(3.905)
structure = Structure(
    lattice,
    ['Sr', 'Ti', 'O', 'O', 'O'],
    [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5],
     [0.5, 0.5, 0.0], [0.5, 0.0, 0.5], [0.0, 0.5, 0.5]],
)

cutoff = 5.0
centres, points, _, distances = structure.get_neighbor_list(
    r=cutoff, exclude_self=True)
Z = np.array(structure.atomic_numbers, dtype=np.int64)
edge_index = np.stack([centres, points], axis=0)
print(f'{len(Z)} atoms, {edge_index.shape[1]} directed edges')


## Expand distances in a Gaussian basis

Raw scalar distances make poor neural-network inputs. CGCNN expands each edge distance in a fixed bank of Gaussians, turning one number into a smooth feature vector.

In [ ]:
import torch.nn as nn

class GaussianBasis(nn.Module):
    def __init__(self, r_min=0.0, r_max=8.0, n_basis=64):
        super().__init__()
        centres = torch.linspace(r_min, r_max, n_basis)
        self.register_buffer('centres', centres)
        self.sigma = float(centres[1] - centres[0])

    def forward(self, r):
        delta = r.unsqueeze(-1) - self.centres
        return torch.exp(-0.5 * (delta / self.sigma) ** 2)

basis = GaussianBasis(0.0, 8.0, 64)
r = torch.tensor(distances, dtype=torch.float32)
edge_attr = basis(r)
print('edge feature tensor:', edge_attr.shape)


## Wrap it in a PyTorch Geometric `Data` object

`Data` is the standard container: node features, edge index, edge features, and (in training) a target. A `batch` vector of zeros marks every atom as belonging to graph 0.

In [ ]:
from torch_geometric.data import Data

graph = Data(
    Z=torch.tensor(Z, dtype=torch.long),
    edge_index=torch.tensor(edge_index, dtype=torch.long),
    edge_attr=edge_attr,
    batch=torch.zeros(len(Z), dtype=torch.long),
)
print(graph)


## Define a small CGCNN-style model

A faithful, compact re-implementation of the Crystal Graph Convolutional Neural Network (Xie & Grossman, 2018): an element embedding, a stack of gated message-passing layers, a global pool, and a small read-out head that produces one scalar per crystal.

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing, global_mean_pool

class CGCNNConv(MessagePassing):
    def __init__(self, atom_dim, edge_dim):
        super().__init__(aggr='add')
        z_dim = 2 * atom_dim + edge_dim
        self.gate_linear = nn.Linear(z_dim, atom_dim)
        self.core_linear = nn.Linear(z_dim, atom_dim)
        self.bn_msg = nn.BatchNorm1d(atom_dim)
        self.bn_out = nn.BatchNorm1d(atom_dim)

    def forward(self, h, edge_index, e):
        agg = self.propagate(edge_index, h=h, e=e)
        return self.bn_out(h + agg)

    def message(self, h_i, h_j, e):
        z = torch.cat([h_i, h_j, e], dim=-1)
        gate = torch.sigmoid(self.gate_linear(z))
        core = F.softplus(self.core_linear(z))
        return self.bn_msg(gate * core)

class CGCNN(nn.Module):
    def __init__(self, n_elements=100, atom_dim=64, edge_dim=64,
                 n_conv=3, hidden_dim=128, n_targets=1):
        super().__init__()
        self.embedding = nn.Embedding(n_elements, atom_dim)
        self.convs = nn.ModuleList(
            [CGCNNConv(atom_dim, edge_dim) for _ in range(n_conv)])
        self.head = nn.Sequential(
            nn.Linear(atom_dim, hidden_dim), nn.Softplus(),
            nn.Linear(hidden_dim, n_targets))

    def forward(self, data):
        h = self.embedding(data.Z)
        for conv in self.convs:
            h = conv(h, data.edge_index, data.edge_attr)
        h_G = global_mean_pool(h, data.batch)
        return self.head(h_G).squeeze(-1)

model = CGCNN(edge_dim=edge_attr.shape[1])
print(model)


## Run one forward pass

With an untrained model the number is meaningless — but a clean forward pass confirms the graph, the basis expansion and the message-passing layers all fit together. Training this model on Materials Project data is the subject of the chapter's pipeline section.

In [ ]:
model.eval()
with torch.no_grad():
    prediction = model(graph)
print('predicted property (untrained):', float(prediction))
